# Seaborn

## Preliminary Note on the Data

The data is large: 10.8 million transactions, more than 4 GiB in memory for the whole file. We'll only load the columns the exercises need, about 1 GiB, and we'll define one or more functions per exercise to create our charts.

This way, when each function returns, the temporary variables it used are deleted and the memory can be reused.

## Imports

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import time

## Loading the Data

In this lab, we'll use real estate transaction data for all of France, from January 2014 to June 2022: 2022 only covers six months. This data covers neither Alsace nor Moselle (departments 67, 68 and 57), nor Mayotte (976).

This data comes from the [French open data](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/) website. We'll use a reworked version that groups transactions, from the [NyxAether/DVF](https://github.com/NyxAether/DVF) repository on GitHub.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh dvf

Load the data with `pd.read_parquet`, reading only the columns of the `useful_columns` list: the whole file would take more than four times as much memory.

Then create a `date` column from the `annee_mutation`, `mois_mutation` and `jour_mutation` columns with `pd.to_datetime`, and drop these three columns to save memory.

Finally, display the column types of the `DataFrame` and its memory size, strings included (the index, `id_mutation`, holds some). You should get 10,839,063 rows, 11 columns and "memory usage: 1.0 GB".

**Useful documentation**

- [`pandas.read_parquet`](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html)
- [`pandas.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [`pandas.DataFrame.drop`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html)
- [`pandas.DataFrame.info`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html)

In [ ]:
useful_columns = [
  "annee_mutation",
  "mois_mutation",
  "jour_mutation",
  "code_departement",
  "valeur_fonciere",
  "nombre_lots",
  "nombre_pieces_principales",
  "nombre_maisons",
  "nombre_appartements",
  "surface_reelle_bati_total",
  "surface_reelle_bati_maisons",
  "surface_reelle_bati_appartements",
  "surface_terrain_total",
]

# Your code here

### Additional hints

`pd.read_parquet` takes the list of the columns to read through its `columns` argument. `pd.to_datetime` accepts a dictionary whose `year`, `month` and `day` keys give the columns of the year, the month and the day. `df.drop(columns=[...])` returns a new `DataFrame`, without these columns: assign it to `df`. `df.info()` doesn't count the space the strings take (its size then ends with "+"); `df.info(memory_usage="deep")` measures it.

```python
df = pd.read_parquet("dvf/dvf-linearized-2014-2022.parquet", columns=...)
df["date"] = pd.to_datetime(dict(year=df.annee_mutation, month=..., day=...))
df = df.drop(columns=[...])
df.info(memory_usage=...)
```

### Solution

In [ ]:
df = pd.read_parquet("dvf/dvf-linearized-2014-2022.parquet", columns=useful_columns)
df["date"] = pd.to_datetime(
  dict(year=df.annee_mutation, month=df.mois_mutation, day=df.jour_mutation)
)
df = df.drop(columns=["annee_mutation", "mois_mutation", "jour_mutation"])
df.info(memory_usage="deep")

## Number of Transactions per Year: Aggregate Before Plotting

To start, we'll count the transactions (rows of the dataset) per year, in two ways, and compare what they cost.

1. In `count_from_rows`, draw a sample of one million transactions with `df.sample`, then plot their count per year with `sns.countplot`, from the `dt.year` attribute of the `date` column: seaborn receives every row and counts them itself. Measure how long the plot takes with `time.perf_counter`.
2. In `count_aggregated`, first count the transactions of each year, over all the rows, with `value_counts`, then plot these few numbers with `sns.barplot`, and measure the time likewise. Give the figure and its axes a title; `sns.despine` removes the top and right borders.

You should find 1,537,648 transactions in 2021. How long would `countplot` take on the 10.8 million rows? What do you notice about 2022? Explain it with the description of the data, above.

**Useful documentation**

- [`pandas.DataFrame.sample`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html)
- [`time.perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter)
- [`pandas.Series.dt.year`](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.year.html)
- [`pandas.Series.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)
- [`seaborn.despine`](https://seaborn.pydata.org/generated/seaborn.despine.html)

In [ ]:
def count_from_rows() -> None:
  pass  # Your code here


def count_aggregated() -> None:
  pass  # Your code here


count_from_rows()
count_aggregated()

### Additional hints

`df.sample(1_000_000, random_state=0)` draws one million rows at random, the same ones at each run. `time.perf_counter()` returns an instant in seconds: the duration is the difference of two calls. `sns.countplot` counts the rows itself: pass it the series of the years through `x`, and the `Axes` to draw on through `ax`. `df.date.dt.year.value_counts()` returns a series indexed by the years, which `sort_index()` puts back in order; `sns.barplot` then receives its index through `x` and its values through `y`, a single value per bar. `ax.set_title`, `ax.set_xlabel` and `ax.set_ylabel` name the figure and its axes. The `countplot` of one million rows takes several seconds; on the 10.8 million, it would take more than a minute, at each attempt.

```python
def count_from_rows() -> None:
  sample = df.sample(..., random_state=0)
  start = time.perf_counter()
  fig, ax = plt.subplots()
  sns.countplot(x=..., ax=ax)
  ax.set_title(...)
  plt.show()
  duration = time.perf_counter() - start
  print(f"countplot on one million rows: {duration:.1f} s")


def count_aggregated() -> None:
  start = time.perf_counter()
  counts = df.date.dt.year.value_counts().sort_index()
  print(counts)
  fig, ax = plt.subplots()
  sns.barplot(x=..., y=..., ax=ax)
  sns.despine(fig)
  ...
  plt.show()
  print(f"value_counts then barplot: {time.perf_counter() - start:.1f} s")
```

### Solution

In [ ]:
def count_from_rows() -> None:
  sample = df.sample(1_000_000, random_state=0)
  start = time.perf_counter()
  fig, ax = plt.subplots()
  sns.countplot(x=sample.date.dt.year, ax=ax)
  ax.set_title("Transactions per year, on a sample of one million")
  plt.show()
  duration = time.perf_counter() - start
  print(f"countplot on one million rows: {duration:.1f} s")
  # The time grows with the number of rows: about ten times more for all of them
  print(f"so about {duration * len(df) / 1_000_000:.0f} s for the {len(df)} rows")


def count_aggregated() -> None:
  start = time.perf_counter()
  # 10.8 million rows reduced to 9 numbers before plotting
  counts = df.date.dt.year.value_counts().sort_index()
  print(counts)
  fig, ax = plt.subplots()
  sns.barplot(x=counts.index, y=counts.values, ax=ax)
  sns.despine(fig)
  ax.set_title("Transaction count per year")
  ax.set_xlabel("Year")
  ax.set_ylabel("Number of transactions")
  ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter(places=1))
  # 2022 only covers January to June: its bar doesn't compare with the others
  plt.show()
  print(f"value_counts then barplot: {time.perf_counter() - start:.1f} s")


count_from_rows()
count_aggregated()

## Total Property Value by Department

In this exercise, we'll group our data by the values of a column (here the department of the transaction) and display the total property value exchanged per group.

- Group the data by department (`code_departement`) with `groupby`, and compute the sum of `valeur_fonciere` of each group.
- Display these sums with `sns.barplot`, from the department with the most to the one with the least.

In [ ]:
def value_by_department() -> None:
  pass  # Your code here


value_by_department()

### Additional hints

`code_departement` is a categorical column: `groupby(..., observed=True)` only keeps the departments present in the data. `sort_values(ascending=False)` sorts the sums from the largest to the smallest, and the `order` argument of `sns.barplot` gives the order of the bars. With a series, `x` takes its index (the departments) and `y` its values; `figsize=(30, 10)` widens the figure so that the 97 departments can be read.

```python
def value_by_department() -> None:
  values = (
    df.groupby(df.code_departement, observed=True)
    .valeur_fonciere.sum()
    .sort_values(...)
  )

  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=values.index, y=values, order=..., ax=ax)
  ...
  plt.show()
```

### Solution

In [ ]:
def value_by_department() -> None:
  values = (
    df.groupby(df.code_departement, observed=True)
    .valeur_fonciere.sum()
    .sort_values(ascending=False)
  )

  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=values.index, y=values, order=values.index, ax=ax)
  sns.despine(ax=ax)
  ax.set_title("Total transaction amount per department")
  ax.set_xlabel("Department")
  ax.set_ylabel("Total transaction amount")
  ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter(unit="€", places=1))
  plt.show()


value_by_department()

## Total Property Value by Region and by Year

In this part, we'll study how to break down a categorical variable by another variable: we'll produce a chart of the total property value exchanged by region, and this value will itself be broken down by year.

To do so, let's start by fetching a dataset that will let us group departments into regions:

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/france-departements/departements-region.csv -O regions.csv

The following function returns a `DataFrame` that will make it easy to join the region to our working `DataFrame`: its index is the department code, written as in our data ("1", not "01"), and its `region_name` column gives the name of the region.

In [ ]:
def read_regions() -> pd.DataFrame:
  regions = pd.read_csv(
    "regions.csv",
    header=0,
    names=["code_departement", "department_name", "region_name"],
    usecols=["code_departement", "region_name"],
    index_col="code_departement",
    dtype=dict(code_departement=str, region_name="category"),
  )
  # DVF writes the departments without a leading zero ("1", not "01")
  return regions.rename(index=lambda code: code.lstrip("0"))

- Add the name of the region to each transaction with `DataFrame.join`, then check that the `region_name` column has no missing value (`isna().sum()` is 0): `groupby` would silently drop a transaction with no region.
- Group the data by region then by year with `groupby`, and compute the sum of `valeur_fonciere` of each group.
- Display these sums with `sns.barplot`, one bar per year in each region (`hue` argument). Seaborn's tutorial covers such categorical data broken down by another variable.

**Useful documentation**

- [`pandas.DataFrame.join`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html)
- [Seaborn tutorial: categorical data](https://seaborn.pydata.org/tutorial/introduction.html#plots-for-categorical-data)

In [ ]:
def value_by_region_by_year() -> None:
  pass  # Your code here


value_by_region_by_year()

### Additional hints

`df.join(read_regions(), on="code_departement")` matches each transaction with the row of `read_regions()` whose index is its `code_departement`. `groupby` takes a list of keys: the `region_name` column and the `date.dt.year` series, renamed `year` with `rename`. `reset_index()` turns these keys back into columns, which `sns.barplot(data, x=..., y=..., hue=...)` expects. The year is an integer: seaborn would treat it as a continuous variable (a gradient palette, a legend that skips years); `astype(str)` makes it a category. `ax.tick_params(axis="x", rotation=45)` tilts the region names, and `ax.get_legend().set_title(...)` gives the legend a title.

```python
def value_by_region_by_year() -> None:
  with_regions = df.join(read_regions(), on="code_departement")
  print("Transactions with no region:", ...)
  region_order = (
    with_regions.groupby(with_regions.region_name, observed=True)
    .valeur_fonciere.sum()
    .sort_values(ascending=False)
    .index
  )
  data = (
    with_regions.groupby(
      [with_regions.region_name, with_regions.date.dt.year.rename("year")],
      observed=True,
    )
    .valeur_fonciere.sum()
    .reset_index()
  )
  data["year"] = ...

  fig, ax = plt.subplots(figsize=(20, 10))
  sns.barplot(data, x=..., y=..., hue=..., order=region_order, ax=ax)
  ...
  plt.show()
```

### Solution

In [ ]:
def value_by_region_by_year() -> None:
  with_regions = df.join(read_regions(), on="code_departement")
  print("Transactions with no region:", with_regions.region_name.isna().sum())
  region_order = (
    with_regions.groupby(with_regions.region_name, observed=True)
    .valeur_fonciere.sum()
    .sort_values(ascending=False)
    .index
  )
  data = (
    with_regions.groupby(
      [with_regions.region_name, with_regions.date.dt.year.rename("year")],
      observed=True,
    )
    .valeur_fonciere.sum()
    .reset_index()
  )
  data["year"] = data.year.astype(str)

  fig, ax = plt.subplots(figsize=(20, 10))
  sns.barplot(
    data, x="region_name", y="valeur_fonciere", hue="year", order=region_order, ax=ax
  )
  sns.despine(fig)
  ax.set_title("Total transaction amount per region and per year")
  ax.set_xlabel("Region")
  ax.tick_params(axis="x", rotation=45)
  ax.set_ylabel("Total transaction amount")
  ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter(unit="€", places=1))
  ax.get_legend().set_title("Year")
  plt.show()


value_by_region_by_year()

## Pearson and Spearman Correlations

Pearson's correlation measures a linear relationship. Spearman's, the Pearson correlation of the ranks, measures a monotonic relationship (increasing or decreasing), regardless of the gaps between the values. We'll compare them on the 2021 transactions, between the property value, the areas and the number of rooms (the `variables` list).

- Compute the two correlation matrices of these variables with `DataFrame.corr` and its `method` argument.
- Display them side by side, on the two `Axes` of one figure, with `sns.heatmap`, each cell annotated with its value.
- Compare the `valeur_fonciere` row in the two matrices: with `surface_reelle_bati_total`, you should find 0.10 (Pearson) and 0.55 (Spearman).
- Display the five highest property values with `nlargest`, then compute the Pearson correlation between `valeur_fonciere` and `surface_reelle_bati_total` again, without the 1% most expensive transactions (`quantile`).

What explains the gap between the two matrices on the `valeur_fonciere` row?

**Useful documentation**

- [`pandas.DataFrame.corr`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html)
- [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html)
- [`pandas.Series.nlargest`](https://pandas.pydata.org/docs/reference/api/pandas.Series.nlargest.html)
- [`pandas.Series.quantile`](https://pandas.pydata.org/docs/reference/api/pandas.Series.quantile.html)
- [`pandas.Series.corr`](https://pandas.pydata.org/docs/reference/api/pandas.Series.corr.html)

In [ ]:
variables = [
  "valeur_fonciere",
  "surface_reelle_bati_total",
  "surface_reelle_bati_maisons",
  "surface_reelle_bati_appartements",
  "surface_terrain_total",
  "nombre_pieces_principales",
]


def correlations() -> None:
  pass  # Your code here


correlations()

### Additional hints

`df.loc[df.date.dt.year == 2021, variables]` keeps the 2021 transactions and the columns of the list. `corr(method="spearman")` computes Spearman's correlation (`"pearson"` by default). `fig, axes = plt.subplots(1, 2, figsize=(18, 7))` creates a figure of two `Axes`, `axes[0]` and `axes[1]`, which `sns.heatmap` receives through `ax`. `annot=True` writes the value in each cell, `fmt=".2f"` with two decimals; `vmin=-1, vmax=1` set the color scale, and a diverging palette (`cmap="vlag"`) shows its sign. `quantile(0.99)` returns the value below which 99% of the transactions fall, and `series.corr(other_series)` the correlation of two columns.

```python
def correlations() -> None:
  sales_2021 = df.loc[df.date.dt.year == 2021, variables]
  fig, axes = plt.subplots(1, 2, figsize=(18, 7))
  for ax, method in zip(axes, ["pearson", "spearman"]):
    sns.heatmap(
      sales_2021.corr(method=...),
      vmin=-1,
      vmax=1,
      cmap="vlag",
      annot=...,
      fmt=".2f",
      ax=ax,
    )
    ax.set_title(f"Correlations ({method}, 2021)")
  fig.tight_layout()
  plt.show()

  print(sales_2021.valeur_fonciere.nlargest(...))
  threshold = sales_2021.valeur_fonciere.quantile(...)
  without_extremes = sales_2021[sales_2021.valeur_fonciere <= threshold]
  ...
```

### Solution

In [ ]:
def correlations() -> None:
  sales_2021 = df.loc[df.date.dt.year == 2021, variables]
  fig, axes = plt.subplots(1, 2, figsize=(18, 7))
  for ax, method in zip(axes, ["pearson", "spearman"]):
    sns.heatmap(
      sales_2021.corr(method=method),
      vmin=-1,
      vmax=1,
      cmap="vlag",
      annot=True,
      fmt=".2f",
      ax=ax,
    )
    ax.set_title(f"Correlations ({method}, 2021)")
  fig.tight_layout()
  plt.show()

  print(sales_2021.valeur_fonciere.nlargest(5))
  threshold = sales_2021.valeur_fonciere.quantile(0.99)
  without_extremes = sales_2021[sales_2021.valeur_fonciere <= threshold]
  pearson = without_extremes.valeur_fonciere.corr(
    without_extremes.surface_reelle_bati_total
  )
  print(
    f"Pearson without the most expensive 1% (above {threshold:.0f} €): {pearson:.2f}"
  )
  # A few huge sales (up to 620 million euros: buildings, sets of lots) dominate
  # the variance of valeur_fonciere, hence Pearson, which falls to 0.10. Spearman
  # only sees their rank: 0.55. Without the most expensive 1%, Pearson rises back
  # to 0.37. Between the total built area and the number of rooms, with no values
  # that extreme, the two agree (0.92 and 0.96).


correlations()

## House and Apartment Prices

In this exercise, we'll compute the price per square meter of apartments and houses separately, and display how it changes month by month.

- For apartments, only keep the rows of the `DataFrame` where:
    - `nombre_appartements`, `valeur_fonciere` and `surface_reelle_bati_appartements` are strictly positive;
    - `nombre_maisons` is zero.
- In the same way, for houses, only keep the rows where:
    - `nombre_maisons`, `valeur_fonciere` and `surface_reelle_bati_maisons` are strictly positive;
    - `nombre_appartements` is zero.
- Compute the price per square meter of each kept transaction (`valeur_fonciere` divided by the area), then the median of these prices for each month. Large sales (a whole building, dozens of lots) would pull a mean or a ratio of sums, not the median.
- Display the two curves on one figure with `sns.lineplot`, one color per property type (`hue` argument), the vertical axis titled "Median price per m²".

For apartments, in June 2022, you should find 3,723 €/m² (2,051 €/m² for houses).

**Useful documentation**

- [`pandas.Series.dt.to_period`](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.to_period.html)
- [`pandas.concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)
- [`pandas.DataFrame.melt`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.melt.html)
- [`seaborn.lineplot`](https://seaborn.pydata.org/generated/seaborn.lineplot.html)

In [ ]:
def houses_apartments_prices() -> None:
  pass  # Your code here


houses_apartments_prices()

### Additional hints

A mask combines its conditions with `&`, each in parentheses: `(df.nombre_maisons > 0) & (df.nombre_appartements == 0)`. `date.dt.to_period("M")` brings each date down to its month, and `.dt.to_timestamp()` turns it back into a date (the first of the month), which `sns.lineplot` puts on a time axis; `price_per_m2.groupby(month).median()` then gives the median of each month. `sns.lineplot` expects a single `DataFrame` in long format, one row per month and property type: `pd.concat([...], axis=1)` puts the two series side by side (each named with `rename`), `reset_index()` makes the month a column, and `melt` stacks the two price columns into one, next to a column that gives the property type.

```python
def houses_apartments_prices() -> None:
  mask_apartments = (df.nombre_appartements > 0) & (df.nombre_maisons == 0) & ...
  mask_houses = ...

  def median_price_per_m2(mask: pd.Series, area_column: str) -> pd.Series:
    sales = df[mask]
    price_per_m2 = sales.valeur_fonciere / sales[area_column]
    month = sales.date.dt.to_period("M").dt.to_timestamp()
    return price_per_m2.groupby(month).median()

  data = pd.concat(
    [
      median_price_per_m2(mask_houses, ...).rename("houses"),
      median_price_per_m2(mask_apartments, ...).rename("apartments"),
    ],
    axis=1,
  )
  data = data.reset_index().melt(
    id_vars="date", var_name="property_type", value_name="price_per_m2"
  )

  fig, ax = plt.subplots()
  sns.lineplot(data, x=..., y=..., hue=..., ax=ax)
  ...
  plt.show()
```

### Solution

In [ ]:
def houses_apartments_prices() -> None:
  mask_apartments = (
    (df.nombre_appartements > 0)
    & (df.nombre_maisons == 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_appartements > 0)
  )
  mask_houses = (
    (df.nombre_appartements == 0)
    & (df.nombre_maisons > 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_maisons > 0)
  )

  def median_price_per_m2(mask: pd.Series, area_column: str) -> pd.Series:
    sales = df[mask]
    price_per_m2 = sales.valeur_fonciere / sales[area_column]
    month = sales.date.dt.to_period("M").dt.to_timestamp()
    return price_per_m2.groupby(month).median()

  data = pd.concat(
    [
      median_price_per_m2(mask_houses, "surface_reelle_bati_maisons").rename("houses"),
      median_price_per_m2(mask_apartments, "surface_reelle_bati_appartements").rename(
        "apartments"
      ),
    ],
    axis=1,
  )
  data = data.reset_index().melt(
    id_vars="date", var_name="property_type", value_name="price_per_m2"
  )

  fig, ax = plt.subplots()
  sns.lineplot(data, x="date", y="price_per_m2", hue="property_type", ax=ax)
  sns.despine(fig)
  ax.tick_params(axis="x", rotation=45)
  ax.set_xlabel("Month")
  ax.set_ylabel("Median price per m²")
  ax.set_title("Median price per square meter")
  ax.set_ylim(bottom=0)
  ax.get_yaxis().set_major_formatter(
    matplotlib.ticker.EngFormatter(unit="€/m²", places=1)
  )
  ax.get_legend().set_title("Property type")
  plt.show()


houses_apartments_prices()

## Apartments Sold in 2021 (Optional)

This optional exercise draws three plots the previous exercises don't use: the histogram, the violin plot and the scatter plot. It's about the sales of a single apartment in 2021: `nombre_appartements` is 1, `nombre_maisons` is zero, `valeur_fonciere` and `surface_reelle_bati_appartements` are strictly positive.

1. Select these sales, add their region (`read_regions`, as above) and a `price_per_m2` column, the property value divided by the area. You should find 418,559 of them, with a median price of €162,000.
2. Plot the histogram of `valeur_fonciere` with `sns.histplot`, on a logarithmic scale. What does it show that a histogram on a linear scale would squash?
3. Plot the distribution of the price per m² of each region with `sns.violinplot`, one region per row, from the most to the least expensive (median price), on the sales whose price per m² lies between the 1st and the 99th percentile. What does the Île-de-France violin show?
4. Plot the property value against the area with `sns.scatterplot`, on a sample of 5,000 sales, with transparent points and both axes on a logarithmic scale. Why a sample?

**Useful documentation**

- [`seaborn.histplot`](https://seaborn.pydata.org/generated/seaborn.histplot.html)
- [`pandas.Series.between`](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html)
- [`seaborn.scatterplot`](https://seaborn.pydata.org/generated/seaborn.scatterplot.html)

In [ ]:
def apartments_2021() -> None:
  pass  # Your code here


apartments_2021()

### Additional hints

The mask follows the one of the previous exercise, with `df.date.dt.year == 2021` and `df.nombre_appartements == 1`. `log_scale=True` puts the axis of `sns.histplot` on a logarithmic scale, and its bins with it. `quantile([0.01, 0.99])` returns the two percentiles, and `series.between(low, high)` keeps the values between them. With `x="price_per_m2", y="region_name"`, `sns.violinplot` draws one violin per region, lying down; `order` receives the regions sorted by median price, and `cut=0` stops each violin at its data. `sample(5_000, random_state=0)` draws the sample, `alpha=0.3` makes the points transparent, and `ax.set(xscale="log", yscale="log")` puts both axes on a logarithmic scale.

```python
def apartments_2021() -> None:
  mask = (df.date.dt.year == 2021) & (df.nombre_appartements == 1) & ...
  apartments = df[mask].join(read_regions(), on="code_departement")
  apartments["price_per_m2"] = ...
  print(len(apartments), apartments.valeur_fonciere.median())

  fig, ax = plt.subplots()
  sns.histplot(apartments, x="valeur_fonciere", log_scale=..., ax=ax)
  ...
  plt.show()

  low, high = apartments.price_per_m2.quantile([0.01, 0.99])
  plausible = apartments[apartments.price_per_m2.between(low, high)]
  order = (
    plausible.groupby("region_name", observed=True)
    .price_per_m2.median()
    .sort_values(ascending=False)
    .index
  )
  fig, ax = plt.subplots(figsize=(10, 8))
  sns.violinplot(plausible, x=..., y=..., order=order, cut=0, ax=ax)
  ...
  plt.show()

  sample = apartments.sample(..., random_state=0)
  fig, ax = plt.subplots()
  sns.scatterplot(sample, x=..., y=..., alpha=0.3, ax=ax)
  ax.set(xscale="log", yscale="log")
  ...
  plt.show()
```

### Solution

In [ ]:
def apartments_2021() -> None:
  mask = (
    (df.date.dt.year == 2021)
    & (df.nombre_appartements == 1)
    & (df.nombre_maisons == 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_appartements > 0)
  )
  apartments = df[mask].join(read_regions(), on="code_departement")
  apartments["price_per_m2"] = (
    apartments.valeur_fonciere / apartments.surface_reelle_bati_appartements
  )
  print(len(apartments), apartments.valeur_fonciere.median())

  fig, ax = plt.subplots()
  sns.histplot(apartments, x="valeur_fonciere", log_scale=True, ax=ax)
  sns.despine(fig)
  ax.set_title("Prices of the apartments sold in 2021")
  ax.set_xlabel("Property value (€, logarithmic scale)")
  ax.set_ylabel("Number of sales")
  # Half the sales between 102,000 and 255,000 € (the quartiles), and a tail of
  # sales below 1,000 € (447), which on a linear scale the sale at nearly 90
  # million would squash against the axis
  plt.show()

  low, high = apartments.price_per_m2.quantile([0.01, 0.99])
  plausible = apartments[apartments.price_per_m2.between(low, high)]
  order = (
    plausible.groupby("region_name", observed=True)
    .price_per_m2.median()
    .sort_values(ascending=False)
    .index
  )
  fig, ax = plt.subplots(figsize=(10, 8))
  sns.violinplot(
    plausible, x="price_per_m2", y="region_name", order=order, cut=0, ax=ax
  )
  sns.despine(fig)
  ax.set_title("Price per m² of the apartments sold in 2021, by region")
  ax.set_xlabel("Price per m² (€)")
  ax.set_ylabel("Region")
  # Île-de-France has the widest violin, up to more than 10,000 €/m²: Paris and
  # its suburbs don't have the same market
  plt.show()

  # The 418,559 points would overlap in one blob, slow to draw
  sample = apartments.sample(5_000, random_state=0)
  fig, ax = plt.subplots()
  sns.scatterplot(
    sample,
    x="surface_reelle_bati_appartements",
    y="valeur_fonciere",
    alpha=0.3,
    ax=ax,
  )
  ax.set(xscale="log", yscale="log")
  sns.despine(fig)
  ax.set_title("Property value and area (5,000 sales in 2021)")
  ax.set_xlabel("Area (m²)")
  ax.set_ylabel("Property value (€)")
  plt.show()


apartments_2021()